In [0]:
# ================================================================
# NOTEBOOK: nb_silver_returns_initial
# PURPOSE:  One-time full load Bronze → Silver for Returns
# RUN:      ONCE only — never run again after first execution
# SOURCE:   bronze/returns/  (parquet from ADF)
# TARGET:   silver/returns/  (Delta format)
# ================================================================



from pyspark.sql import functions as F
from pyspark.sql.functions import col, trim, when,to_timestamp, upper, trim, round
from pyspark.sql.window import Window

BRONZE_PATH = "abfss://source@stshopsensedevhj.dfs.core.windows.net/bronze/returns/"
SILVER_PATH = "abfss://source@stshopsensedevhj.dfs.core.windows.net/silver/returns/"

# Read full Bronze
bronze_df = spark.read.parquet(BRONZE_PATH)
print(f"[BRONZE] ROWS read: {bronze_df.count()} rows")
bronze_df.printSchema()

# ── STEP 1: Deduplication on ReturnID ────────────────────────

dedup_window = Window.partitionBy("ReturnID").orderBy(F.desc("LastModifiedDate")) 
bronze_df = (
    bronze_df
    .withColumn("_rn", F.row_number().over(dedup_window))
    .filter(col("_rn") ==1)
    .drop("_rn")
)

# ── STEP 2: Remove nulls on key columns ──────────────────
silver_df = (
bronze_df
.filter(col("ReturnID").isNotNull())
.filter(col("OrderID").isNotNull())
.filter(col("CustomerID").isNotNull())
.filter(col("ReturnDate").isNotNull())
)

# ── STEP 3: Type casting ──────────────────────────────────────
silver_df = (
bronze_df
.withColumn("ReturnDate",   to_timestamp("ReturnDate"))
.withColumn("RefundDate",     to_timestamp("RefundDate"))
.withColumn("LastModifiedDate",     to_timestamp("LastModifiedDate"))
.withColumn("RefundAmount",     col("RefundAmount").cast("decimal(10,2)"))
)
# ── STEP 4: Standardize string columns ───────────────────────
silver_df = (
    bronze_df
    .withColumn("ReturnStatus",     upper(trim(col("ReturnStatus"))))
    .withColumn("ReturnReason",     upper(trim(col("ReturnReason"))))
    .withColumn("RefundMethod",     upper(trim(col("RefundMethod"))))
    .withColumn("ConditionOnReturn",  upper(trim(col("ConditionOnReturn"))))
)

# ── STEP 5: Business derived columns ─────────────────────────
silver_df = ( 
    bronze_df
    # Was the return approved / rejected / still pending?
    .withColumn("IsApproved",  col("ReturnStatus") == "REUNDED")
    .withColumn("IsRejected",  col("ReturnStatus") == "REJECTED")
    .withColumn("IsPending",   col("ReturnStatus") == "PROCESSING")

    # Days taken to process return (how fast is ops team?)
    .withColumn("DaysToProcess",    when(col("RefundDate").isNotNull(), F.datediff(col("RefundDate"),col("ReturnDate")))
                .otherwise(None))
    
    # Is this a fast return? (returned within 3 days of delivery)
    # Note: We don't have DeliveredDate here, so use days to process as proxy

    .withColumn("IsQuickReturn",   when(col("DaysToProcess").isNotNull(),col("DaysToProcess") <=3).otherwise(False))


    # Return condition flag (damaged = potential fraud or mishandling)
    .withColumn("IsDamaged",  col("ConditionOnReturn") == "DAMAGED")

    # Return reason category (group similar reasons)
    .withColumn("ReasonCategory", 
        when(col("ReturnReason").isin("DEFECTIVE_PRODUCT","QUALITY_ISSUE"),
                 "QUALITY")
        .when(col("ReturnReason").isin("WRONG_ITEM","NOT_AS_DESCRIBED"),
              "WRONG_PRODUCT")
        .when(col("ReturnReason") == "SIZE_MISMATCH",
              "FIT_ISSUE")
        .when(col("ReturnReason") == "CHANGED_MIND",
              "BUYER_REMORSE")
        .when(col("ReturnReason") == "LATE_DELIVERY",
              "DELIVERY_ISSUE")
        .otherwise("OTHER"))
    
    # Year and Month of return (for trend analysis in Gold)
    .withColumn("ReturnYear", F.year("ReturnDate"))
    .withColumn("ReturnMonth", F.month("ReturnDate"))


    # Metadata
    .withColumn("_silver_load_ts", F.current_timestamp())
    .withColumn("source", F.lit("initial_full_load"))
    .withColumn("_is_deleted",  F.lit(False))

)

    # ── STEP 6: Write Silver as Delta ─────────────────────────────
(
    silver_df.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_PATH)
)

    # ── Verify ───────────────────────────────────────────────────
total = silver_df.count()
approved = silver_df.filter(col("IsApproved")).count()
rejected = silver_df.filter(col("IsRejected")).count()
pending = silver_df.filter(col("IsPending")).count()
damaged = silver_df.filter(col("IsDamaged")).count()


print(f"[DONE]silver/returns/ written: {total} rows")
print(f" Approved(Refunded): {approved}")
print(f" Rejected:            {rejected}")
print(f" Pending:           {pending}")
print(f" Damageditems", damaged)
print("\n[REASON BREAKDOWN]")

silver_df.groupBy("ReasonCategory").count().orderBy("count", ascending=False).show()

display(silver_df.limit(10))


[BRONZE] ROWS read: 404 rows
root
 |-- ReturnID: string (nullable = true)
 |-- OrderID: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- ReturnDate: timestamp (nullable = true)
 |-- ReturnReason: string (nullable = true)
 |-- ReturnStatus: string (nullable = true)
 |-- RefundAmount: decimal(10,2) (nullable = true)
 |-- RefundDate: timestamp (nullable = true)
 |-- RefundMethod: string (nullable = true)
 |-- ConditionOnReturn: string (nullable = true)
 |-- LastModifiedDate: timestamp (nullable = true)

[DONE]silver/returns/ written: 404 rows
 Approved(Refunded): 0
 Rejected:            85
 Pending:           73
 Damageditems 90

[REASON BREAKDOWN]
+--------------+-----+
|ReasonCategory|count|
+--------------+-----+
| WRONG_PRODUCT|  119|
|       QUALITY|  114|
| BUYER_REMORSE|   70|
|     FIT_ISSUE|   52|
|DELIVERY_ISSUE|   49|
+--------------+-----+



ReturnID,OrderID,CustomerID,ReturnDate,ReturnReason,ReturnStatus,RefundAmount,RefundDate,RefundMethod,ConditionOnReturn,LastModifiedDate,IsApproved,IsRejected,IsPending,DaysToProcess,IsQuickReturn,IsDamaged,ReasonCategory,ReturnYear,ReturnMonth,_silver_load_ts,source,_is_deleted
RET000001,ORD0002000,CUST00025,2024-03-12T11:13:51Z,DEFECTIVE_PRODUCT,PROCESSING,20228.10,2024-03-18T11:13:51Z,Wallet,SEALED,2024-03-12T11:13:51Z,false,false,true,6,false,false,QUALITY,2024,3,2026-07-10T21:14:42.323926Z,initial_full_load,false
RET000002,ORD0002520,CUST00499,2024-06-22T05:05:13Z,QUALITY_ISSUE,REFUNDED,3193.91,2024-06-24T05:05:13Z,COD,SEALED,2024-06-22T05:05:13Z,false,false,false,2,true,false,QUALITY,2024,6,2026-07-10T21:14:42.323926Z,initial_full_load,false
RET000003,ORD0002560,CUST00289,2024-06-20T15:14:33Z,SIZE_MISMATCH,REFUNDED,22255.86,2024-06-22T15:14:33Z,UPI,GOOD,2024-06-20T15:14:33Z,false,false,false,2,true,false,FIT_ISSUE,2024,6,2026-07-10T21:14:42.323926Z,initial_full_load,false
RET000004,ORD0000385,CUST00362,2024-05-16T21:48:50Z,CHANGED_MIND,REFUNDED,46927.83,2024-05-21T21:48:50Z,UPI,OPENED,2024-05-16T21:48:50Z,false,false,false,5,false,false,BUYER_REMORSE,2024,5,2026-07-10T21:14:42.323926Z,initial_full_load,false
RET000005,ORD0002647,CUST00330,2024-02-10T04:19:09Z,CHANGED_MIND,REJECTED,26497.14,2024-02-16T04:19:09Z,Wallet,GOOD,2024-02-10T04:19:09Z,false,true,false,6,false,false,BUYER_REMORSE,2024,2,2026-07-10T21:14:42.323926Z,initial_full_load,false
RET000006,ORD0000542,CUST00455,2024-04-07T06:06:58Z,NOT_AS_DESCRIBED,PROCESSING,7026.86,2024-04-13T06:06:58Z,DebitCard,SEALED,2024-04-07T06:06:58Z,false,false,true,6,false,false,WRONG_PRODUCT,2024,4,2026-07-10T21:14:42.323926Z,initial_full_load,false
RET000007,ORD0001294,CUST00147,2024-04-07T18:46:47Z,NOT_AS_DESCRIBED,REJECTED,27812.04,2024-04-13T18:46:47Z,COD,GOOD,2024-04-07T18:46:47Z,false,true,false,6,false,false,WRONG_PRODUCT,2024,4,2026-07-10T21:14:42.323926Z,initial_full_load,false
RET000008,ORD0000794,CUST00081,2024-01-13T12:39:35Z,LATE_DELIVERY,REFUNDED,10311.89,2024-01-18T12:39:35Z,COD,GOOD,2024-01-13T12:39:35Z,false,false,false,5,false,false,DELIVERY_ISSUE,2024,1,2026-07-10T21:14:42.323926Z,initial_full_load,false
RET000009,ORD0002906,CUST00192,2024-01-12T16:35:04Z,SIZE_MISMATCH,REFUNDED,47252.11,2024-01-15T16:35:04Z,COD,SEALED,2024-01-12T16:35:04Z,false,false,false,3,true,false,FIT_ISSUE,2024,1,2026-07-10T21:14:42.323926Z,initial_full_load,false
RET000010,ORD0002236,CUST00100,2024-02-22T07:49:31Z,SIZE_MISMATCH,REFUNDED,28487.90,2024-02-28T07:49:31Z,UPI,SEALED,2024-02-22T07:49:31Z,false,false,false,6,false,false,FIT_ISSUE,2024,2,2026-07-10T21:14:42.323926Z,initial_full_load,false
